# Class Lab: Discover Association Rules by Apriori Algorithm

## Step 1: setup goal
- use Apriori algorithm to discover association riles

## Step 2: retrieve data
use dummy data for demonstration

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# prepare transaction
transactions = [
    ['Milk', 'Bread', 'Butter'],   # T1
    ['Beer', 'Bread'],             # T2
    ['Milk', 'Beer', 'Butter'],    # T3
    ['Milk', 'Bread', 'Beer'],     # T4
    ['Milk', 'Bread', 'Butter'],   # T5
]
transactions

In [5]:
# convert to one-hot encoding
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("One-hot Transaction Table:")
print(df.astype(int), "\n")
df

One-hot Transaction Table:
   Beer  Bread  Butter  Milk
0     0      1       1     1
1     1      1       0     0
2     1      0       1     1
3     1      1       0     1
4     0      1       1     1 



,Beer,Bread,Butter,Milk
0,False,True,True,True
1,True,True,False,False
2,True,False,True,True
3,True,True,False,True
4,False,True,True,True


## Step 3: prepare data

1. Separate data set into features variables (X) and target variable (y)
2. Standardizes all the features in X so they are on the same scale (mean = 0, variance = 1). <br>
   This prevents features with larger numeric ranges (like “total sulfur dioxide”) from dominating smaller ones (like “sulphates”).<br>
   z = (x-μ)/σ

In [ ]:
# y is a vector (series) and represents the target variable of wine quality.
y = df_wine['quality']

# X is a matrix (data frame) of features variables. These features are wine properties such as density and alcohol presence.
X = df_wine[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']]

# standardizes all the features in X, X is a ndarray now
X_scaled = preprocessing.StandardScaler().fit(X).transform(X)
X_scaled[0:3]

## Step 4: explore data

In [ ]:
# explore data from dimension reduction perspective by PCA
pca = PCA()
pca_model = pca.fit(X_scaled)
X_projected = pca_model.transform(X_scaled)
# plt.plot(pca_model.explained_variance_, marker='o', linestyle='-', color='b')

# cumulative explained variance
cumulative_var = np.cumsum(pca_model.explained_variance_ratio_)

plt.plot(cumulative_var, marker='o', linestyle='-', color='b')

for i, val in enumerate(cumulative_var):
    plt.text(i + 0.2, val, f'{val:.0%}', fontsize=8, color='r')

plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance Ratio")
plt.xticks(np.arange(len(cumulative_var)), np.arange(1, len(cumulative_var) + 1))

plt.title("Cumulative Explained Variance by PCA")
plt.grid(True)
plt.show()

In [ ]:
# illustrate how latent variables are constructed
latent_vars = pd.DataFrame(pca_model.components_, columns=list(['fixed acidity', 'volatile acidity', 'citric acid',\
                                                 'residual sugar','chlorides', 'free sulfur dioxide',\
                                                 'total sulfur dioxide', 'density','pH',\
                                                 'sulphates', 'alcohol']))
latent_vars.round(2).head(5)

In [ ]:
# Customize the row indices with letters
latent_var_names = ['PC1_Persistent acidity', 'PC2_Sulfides', 'PC3_Volatile acidity', 'PC4_Chlorides', 'PC5_Lack of residual sugar']
top_5_latent_vars = latent_vars.iloc[:5, :]
top_5_latent_vars.index = latent_var_names
top_5_latent_vars.round(2)

## Step 3: prepare data again (top-5 PCA components)

In [ ]:
# transform raw data into the result of dimensionality reduction after PCA
X_projected_df = pd.DataFrame(X_projected).iloc[:,:5]
X_projected_df.columns = latent_var_names
X_projected_df.head().round(2)

## Step 5: build model
Comparing the accuracy of the original data set with latent variables

In [ ]:
# Wine score prediction before principal component analysis
from sklearn.datasets import load_digits
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (confusion_matrix, accuracy_score)
import matplotlib.pyplot as plt
import math

gnb = GaussianNB()
gnb.fit(X_scaled, y)

# test model
y_pred = gnb.predict(X_scaled)

# validate model - accuracy and confusion_matrix
acc = accuracy_score(y, y_pred)
print(f"Accuracy: {acc:2f} \n")

print(confusion_matrix(y, y_pred),'\n')
print('# of correct predictions:', confusion_matrix(y, y_pred).trace(),'\n') # sum of diagonal elements

In [ ]:
# Wine score prediction with top-5 principal components
pca = PCA(n_components= 5)
pca_model = pca.fit(X_scaled)
X_projected = pca_model.transform(X_scaled)

gnb = GaussianNB()
gnb.fit(X_projected, y)

# test model
y_pred = gnb.predict(X_projected)

# validate model - accuracy and confusion_matrix
acc = accuracy_score(y, y_pred)
print(f"Accuracy: {acc:2f} \n")

print(confusion_matrix(y, y_pred),'\n')
print('# of correct predictions:', confusion_matrix(y, y_pred).trace(),'\n') # sum of diagonal elements

## Step 6: presentation

In [ ]:
gnb = GaussianNB()
predicted_correct = []
for i in range(1,10):
    pca = PCA(n_components= i)
    pca_model = pca.fit(X_scaled)
    X_projected = pca_model.transform(X_scaled)
    gnb.fit(X_projected, y)

    # test model
    y_pred = gnb.predict(X_projected)
    predicted_correct.append(confusion_matrix(y, y_pred).trace())
    print(list(map(int,predicted_correct)))
plt.plot(predicted_correct)
plt.grid()
plt.show()          